# Local Pipeline: Image Denoising + Object Detection

Uses dataset from assets folder

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

sys.path.append('../src')
from data.local_loader import LocalDataLoader

%matplotlib inline

## Load Images from Assets

In [ ]:
loader = LocalDataLoader('assets/pascal-voc')
images = loader.load_sample_images(6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (filename, img) in enumerate(images):
    axes[idx].imshow(img)
    axes[idx].set_title(filename)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"Loaded {len(images)} images from assets")

## Image Denoising

In [ ]:
filename, original = images[0]

gaussian = cv2.GaussianBlur(original, (5, 5), 0)
bilateral = cv2.bilateralFilter(original, 9, 75, 75)
nlm = cv2.fastNlMeansDenoisingColored(original, None, 10, 10, 7, 21)

psnr_gaussian = peak_signal_noise_ratio(original, gaussian)
psnr_bilateral = peak_signal_noise_ratio(original, bilateral)
psnr_nlm = peak_signal_noise_ratio(original, nlm)

print("PSNR Results:")
print(f"Gaussian: {psnr_gaussian:.2f} dB")
print(f"Bilateral: {psnr_bilateral:.2f} dB")
print(f"NLM: {psnr_nlm:.2f} dB")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(original); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(gaussian); axes[1].set_title(f'Gaussian\n{psnr_gaussian:.2f} dB'); axes[1].axis('off')
axes[2].imshow(bilateral); axes[2].set_title(f'Bilateral\n{psnr_bilateral:.2f} dB'); axes[2].axis('off')
axes[3].imshow(nlm); axes[3].set_title(f'NLM\n{psnr_nlm:.2f} dB'); axes[3].axis('off')
plt.tight_layout()
plt.show()

## Object Detection

In [ ]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F
import matplotlib.patches as patches

model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

def detect_objects(image, threshold=0.8):
    img_tensor = F.to_tensor(image).unsqueeze(0).to(device)
    with torch.no_grad():
        predictions = model(img_tensor)[0]
    keep = predictions['scores'] > threshold
    return (
        predictions['boxes'][keep].cpu().numpy(),
        predictions['labels'][keep].cpu().numpy(),
        predictions['scores'][keep].cpu().numpy()
    )

print("Model loaded")

## Compare Detection Results

In [ ]:
boxes_orig, labels_orig, scores_orig = detect_objects(original, threshold=0.8)
boxes_nlm, labels_nlm, scores_nlm = detect_objects(nlm, threshold=0.8)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, img, boxes, labels, scores, title in [
    (axes[0], original, boxes_orig, labels_orig, scores_orig, 'Original'),
    (axes[1], nlm, boxes_nlm, labels_nlm, scores_nlm, 'Denoised')
]:
    ax.imshow(img)
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                            linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{COCO_CLASSES[label]}: {score:.2f}', 
               bbox=dict(facecolor='red', alpha=0.5), fontsize=10, color='white')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"Original: {len(boxes_orig)} detections")
print(f"Denoised: {len(boxes_nlm)} detections")